In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import re
import numpy as np
import pandas as pd

from scipy.stats import chi2_contingency, pointbiserialr

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.dummy import DummyClassifier

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

In [2]:
DATA_PATH = Path("../data/processed/nhanes_merged_adults_final.csv")
OUTPUT_DIR = Path("../data/processed/model_outputs_prediabetes")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "prediabetes"
RANDOM_STATE = 42
TEST_SIZE = 0.20

print("DATA_PATH:", DATA_PATH.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())

DATA_PATH: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\nhanes_merged_adults_final.csv
OUTPUT_DIR: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_prediabetes


In [3]:
df = pd.read_csv(DATA_PATH, low_memory=False)

print("Shape:", df.shape)
print("First 10 columns:")
print(df.columns[:10].tolist())

assert TARGET in df.columns, f"{TARGET} not found in dataframe."

print(df[TARGET].value_counts(dropna=False).sort_index())
print()
print("Target prevalence:")
print(df[TARGET].value_counts(normalize=True, dropna=False).sort_index().round(4))

Shape: (7437, 878)
First 10 columns:
['SEQN', 'age_years', 'income_poverty_ratio', 'mec_exam_weight', 'interview_weight', 'survey_psu', 'survey_stratum', 'gender', 'ethnicity', 'education']
prediabetes
0    6755
1     682
Name: count, dtype: int64

Target prevalence:
prediabetes
0    0.9083
1    0.0917
Name: proportion, dtype: float64


In [4]:
overview = pd.DataFrame({
    "feature": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "missing_pct": df.isna().mean().values * 100,
    "n_unique": [df[c].nunique(dropna=True) for c in df.columns]
}).sort_values(["missing_pct", "n_unique"], ascending=[False, True])

overview.head(30)

,feature,dtype,missing_pct,n_unique
166,mcq149___menstrual_periods_started_yet?,float64,100.000000,0
167,mcq151___age_in_years_at_first_menstrual_period,float64,100.000000,0
294,smq621___cigarettes_smoked_in_entire_life,float64,100.000000,0
295,smd630___age_first_smoked_whole_cigarette,float64,100.000000,0
434,LBXIGG_cytomegalovirus_cmv_igg,float64,100.000000,0
435,LBXIGM_cytomegalovirus_cmv_igm,float64,100.000000,0
436,LBXIGGA_cytomegalovirus_cmv_igg_avidity,float64,100.000000,0
357,medication_21,str,99.973107,2
358,medication_22,str,99.973107,2
356,medication_20,str,99.959661,3


In [5]:
LEAKAGE_COLS = [
    TARGET,
    "diabetes",
    "diq010___doctor_told_you_have_diabetes",
    "diq160___ever_told_you_have_prediabetes",
    "rxd_disease_list",
]

QUESTIONNAIRE_PREFIXES = (
    "slq", "sld", "dpq", "paq", "huq", "mcq", "bpq", "kiq", "cdq", "smq", "alq",
    "whq", "rhq"
)

EXCLUDE_PATTERNS = [
    r"^seqn$",
    r"cluster",
    r"stratum",
    r"weight",
    r"sample_weight",
    r"survey_",
    r"^rxd",
    r"medication",
    r"drug",
    r"icd",
    r"diagnosis",
    r"lab",
    r"exam",
    r"fasting",
    r"liver_",
    r"bmi$",
    r"waist",
    r"hip",
    r"height",
    r"weight_kg",
    r"cholesterol",
    r"creatin",
    r"ferritin",
    r"vitamin",
    r"iron",
    r"protein",
    r"carbs",
    r"fat$",
    r"calories",
    r"pulse_",
    r"fatigue_binary",
    r"fatigue_score",
    r"glucose",
    r"insulin",
    r"hba1c",
    r"a1c",
    r"glyco",
]

MANUAL_DROP_COLS = [
    # bei Bedarf später ergänzen
]

In [6]:
def has_allowed_prefix(col: str, prefixes=QUESTIONNAIRE_PREFIXES) -> bool:
    col_low = col.lower()
    return col_low.startswith(prefixes)

def matches_any_pattern(col: str, patterns) -> bool:
    col_low = col.lower()
    return any(re.search(p, col_low) for p in patterns)

def is_binary_series(s: pd.Series) -> bool:
    vals = set(pd.Series(s.dropna()).unique())
    return vals.issubset({0, 1}) and len(vals) <= 2

def get_example_values(s: pd.Series, n=5):
    vals = s.dropna().unique()[:n]
    return list(vals)

def cramers_v(x: pd.Series, y: pd.Series) -> float:
    table = pd.crosstab(x, y)
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan
    chi2 = chi2_contingency(table)[0]
    n = table.values.sum()
    r, k = table.shape
    denom = min(k - 1, r - 1)
    if denom == 0 or n == 0:
        return np.nan
    return np.sqrt((chi2 / n) / denom)

def safe_pointbiserial(x: pd.Series, y: pd.Series):
    valid = x.notna() & y.notna()
    if valid.sum() < 30:
        return np.nan, np.nan
    try:
        r, p = pointbiserialr(x[valid], y[valid])
        return r, p
    except Exception:
        return np.nan, np.nan

def infer_feature_type(series: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(series):
        nunique = series.nunique(dropna=True)
        if nunique <= 10:
            return "categorical"
        return "numeric"
    return "categorical"

In [7]:
candidate_cols = []

for col in df.columns:
    if col == TARGET:
        continue
    if col in LEAKAGE_COLS:
        continue
    if not has_allowed_prefix(col):
        continue
    if matches_any_pattern(col, EXCLUDE_PATTERNS):
        continue
    if col in MANUAL_DROP_COLS:
        continue
    candidate_cols.append(col)

print("Number of initial candidate questionnaire features:", len(candidate_cols))
print(candidate_cols[:80])

Number of initial candidate questionnaire features: 138
['alq111___ever_had_a_drink_of_any_kind_of_alcohol', 'alq121___past_12_mo_how_often_drink_alcoholic_bev', 'alq130___avg_#_alcoholic_drinks/day___past_12_mos', 'alq142___#_days_have_4_or_5_drinks/past_12_mos', 'alq270___#_times_4_5_drinks_in_2hrs/past_12_mos', 'alq280___#_times_8+_drinks_in_1_day/past_12_mos', 'alq290___#_times_12+_drinks_in_1_day/past_12_mos', 'alq151___ever_have_4/5_or_more_drinks_every_day?', 'alq170___past_30_days_#_times_4_5_drinks_on_an_oc', 'bpq020___ever_told_you_had_high_blood_pressure', 'bpq030___told_had_high_blood_pressure___2+_times', 'bpq040a___taking_prescription_for_hypertension', 'bpq050a___now_taking_prescribed_medicine_for_hbp', 'bpq100d___now_taking_prescribed_medicine', 'cdq001___sp_ever_had_pain_or_discomfort_in_chest', 'cdq002___sp_get_it_walking_uphill_or_in_a_hurry', 'cdq003___during_an_ordinary_pace_on_level_ground', 'cdq004___if_so_does_sp_continue_or_slow_down', 'cdq005___does_standing_r

In [8]:
audit_rows = []

for col in candidate_cols:
    s = df[col]
    audit_rows.append({
        "feature": col,
        "dtype": str(s.dtype),
        "missing_pct": round(s.isna().mean() * 100, 2),
        "n_unique": s.nunique(dropna=True),
        "example_values": str(get_example_values(s, n=5)),
        "is_binary_like": is_binary_series(s),
        "keep_for_model": True,
        "drop_reason": ""
    })

audit_df = pd.DataFrame(audit_rows).sort_values(
    ["missing_pct", "n_unique", "feature"],
    ascending=[False, True, True]
)

audit_df.head(100)

,feature,dtype,missing_pct,n_unique,example_values,is_binary_like,keep_for_model,drop_reason
57,mcq149___menstrual_periods_started_yet?,float64,100.00,0,[],True,True,
58,mcq151___age_in_years_at_first_menstrual_period,float64,100.00,0,[],True,True,
136,smq621___cigarettes_smoked_in_entire_life,float64,100.00,0,[],True,True,
78,mcq230c___what_kind_of_cancer_third_mention,float64,99.93,4,"[np.float64(20.0), np.float64(23.0), np.float6...",False,True,
26,cdq009g___pain_in_left_arm,float64,99.85,1,[np.float64(7.0)],False,True,
22,cdq009c___pain_in_neck,float64,99.84,1,[np.float64(3.0)],False,True,
20,cdq009a___pain_in_right_arm,float64,99.81,1,[np.float64(1.0)],True,True,
99,rhq020___age_range_at_first_menstrual_period,float64,99.68,5,"[np.float64(9.0), np.float64(3.0), np.float64(...",False,True,
102,rhq070___age_range_at_last_menstrual_period,float64,99.61,8,"[np.float64(99.0), np.float64(3.0), np.float64...",False,True,
77,mcq230b___what_kind_of_cancer_second_mention,float64,99.61,15,"[np.float64(37.0), np.float64(16.0), np.float6...",False,True,


In [10]:
MAX_MISSING_PCT = 60.0
MIN_NON_NULL = 500
MIN_VARIANCE_UNIQUE = 2

filtered_audit_df = audit_df.copy()

filtered_audit_df.loc[filtered_audit_df["missing_pct"] > MAX_MISSING_PCT, "keep_for_model"] = False
filtered_audit_df.loc[filtered_audit_df["missing_pct"] > MAX_MISSING_PCT, "drop_reason"] = "too_missing"

for idx, row in filtered_audit_df.iterrows():
    col = row["feature"]
    non_null = df[col].notna().sum()
    if non_null < MIN_NON_NULL:
        filtered_audit_df.at[idx, "keep_for_model"] = False
        filtered_audit_df.at[idx, "drop_reason"] = "too_few_non_null"
    elif row["n_unique"] < MIN_VARIANCE_UNIQUE:
        filtered_audit_df.at[idx, "keep_for_model"] = False
        filtered_audit_df.at[idx, "drop_reason"] = "constant_or_near_constant"

filtered_audit_df["keep_for_model"].value_counts()

keep_for_model
False    69
True     69
Name: count, dtype: int64

In [11]:
MANUAL_DROP_AFTER_AUDIT = [
    # Beispiele, je nach Audit:
    # "huq051___#times_receive_healthcare_over_past_year",
    # "mcq366b___doctor_told_to_increase_exercise",
    # "mcq366c___doctor_told_to_reduce_salt",
    # "mcq366d___doctor_told_to_reduce_fat_in_diet",
]

filtered_audit_df.loc[
    filtered_audit_df["feature"].isin(MANUAL_DROP_AFTER_AUDIT),
    ["keep_for_model", "drop_reason"]
] = [False, "manual_drop"]

final_features = filtered_audit_df.loc[filtered_audit_df["keep_for_model"], "feature"].tolist()

print("Final feature count:", len(final_features))
print(final_features[:80])

Final feature count: 69
['rhq305___had_both_ovaries_removed?', 'paq670___days_moderate_recreational_activities', 'rhq540___ever_use_female_hormones?', 'rhq131___ever_been_pregnant?', 'rhq031___had_regular_periods_in_past_12_months', 'rhq010___age_when_first_menstrual_period_occurred', 'paq625___number_of_days_moderate_work', 'cdq001___sp_ever_had_pain_or_discomfort_in_chest', 'cdq010___shortness_of_breath_on_stairs/inclines', 'alq170___past_30_days_#_times_4_5_drinks_on_an_oc', 'alq142___#_days_have_4_or_5_drinks/past_12_mos', 'alq130___avg_#_alcoholic_drinks/day___past_12_mos', 'alq151___ever_have_4/5_or_more_drinks_every_day?', 'alq121___past_12_mo_how_often_drink_alcoholic_bev', 'kiq046___leak_urine_during_nonphysical_activities', 'kiq480___how_many_times_urinate_in_night?', 'kiq044___urinated_before_reaching_the_toilet?', 'kiq042___leak_urine_during_physical_activities?', 'kiq005___how_often_have_urinary_leakage?', 'dpq040___feeling_tired_or_having_little_energy', 'alq111___ever_ha

In [12]:
final_audit_path = OUTPUT_DIR / "prediabetes_feature_audit_final.csv"
filtered_audit_df.to_csv(final_audit_path, index=False)
print("Saved:", final_audit_path.resolve())

Saved: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_prediabetes\prediabetes_feature_audit_final.csv


In [13]:
model_df = df[[TARGET] + final_features].copy()

print("Model dataframe shape:", model_df.shape)
print("Target missing:", model_df[TARGET].isna().sum())

X = model_df[final_features].copy()
y = model_df[TARGET].copy()

mask = y.notna()
X = X.loc[mask].copy()
y = y.loc[mask].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target distribution:")
print(y_train.value_counts(normalize=True).round(4))

Model dataframe shape: (7437, 70)
Target missing: 0
Train shape: (5949, 69)
Test shape: (1488, 69)
Train target distribution:
prediabetes
0    0.9082
1    0.0918
Name: proportion, dtype: float64


In [14]:
univariate_rows = []

for col in final_features:
    s = X_train[col]
    feature_type = infer_feature_type(s)

    row = {
        "feature": col,
        "feature_type": feature_type,
        "missing_pct_train": round(s.isna().mean() * 100, 2),
        "n_unique_train": s.nunique(dropna=True),
        "association_metric": None,
        "association_value": np.nan,
        "abs_association": np.nan,
        "p_value": np.nan,
        "n_train_non_null": int(s.notna().sum())
    }

    if feature_type == "categorical":
        valid = s.notna() & y_train.notna()
        if valid.sum() >= 30:
            try:
                cv = cramers_v(s[valid], y_train[valid])
                table = pd.crosstab(s[valid], y_train[valid])
                p_val = chi2_contingency(table)[1] if table.shape[0] > 1 and table.shape[1] > 1 else np.nan
                row["association_metric"] = "cramers_v"
                row["association_value"] = cv
                row["abs_association"] = abs(cv) if pd.notna(cv) else np.nan
                row["p_value"] = p_val
            except Exception:
                pass
    else:
        r, p = safe_pointbiserial(s, y_train)
        row["association_metric"] = "pointbiserial"
        row["association_value"] = r
        row["abs_association"] = abs(r) if pd.notna(r) else np.nan
        row["p_value"] = p

    univariate_rows.append(row)

univariate_df = pd.DataFrame(univariate_rows).sort_values(
    ["abs_association", "p_value"],
    ascending=[False, True]
)

univariate_df.head(50)

,feature,feature_type,missing_pct_train,n_unique_train,association_metric,association_value,abs_association,p_value,n_train_non_null
52,mcq366b___doctor_told_to_increase_exercise,categorical,0.00,3,cramers_v,0.175833,0.175833,1.150146e-40,5949
54,mcq366d___doctor_told_to_reduce_fat_in_diet,categorical,0.00,3,cramers_v,0.167165,0.167165,7.969839e-37,5949
53,mcq366c___doctor_told_to_reduce_salt,categorical,0.00,3,cramers_v,0.144598,0.144598,9.775706e-28,5949
40,slq310___usual_wake_time_on_weekdays_or_workdays,categorical,0.67,105,cramers_v,0.136869,0.136869,3.083512e-01,5909
42,slq330___usual_wake_time_on_weekends,categorical,0.62,67,cramers_v,0.116325,0.116325,1.153272e-01,5912
45,bpq020___ever_told_you_had_high_blood_pressure,categorical,0.00,3,cramers_v,0.107724,0.107724,1.021255e-15,5949
39,slq300___usual_sleep_time_on_weekdays_or_workdays,categorical,0.71,64,cramers_v,0.106139,0.106139,3.559609e-01,5907
63,"whq040___like_to_weigh_more,_less_or_same",categorical,0.00,4,cramers_v,0.102125,0.102125,2.148577e-13,5949
41,slq320___usual_sleep_time_on_weekends,categorical,0.62,55,cramers_v,0.100751,0.100751,2.669989e-01,5912
4,rhq031___had_regular_periods_in_past_12_months,categorical,54.78,3,cramers_v,0.091793,0.091793,1.197115e-05,2690


In [15]:
univariate_path = OUTPUT_DIR / "prediabetes_univariate_ranking.csv"
univariate_df.to_csv(univariate_path, index=False)
print("Saved:", univariate_path.resolve())

Saved: C:\Users\Philipp\AIBootcamp\halfFull\data\processed\model_outputs_prediabetes\prediabetes_univariate_ranking.csv


In [16]:
numeric_features = []
categorical_features = []

for col in final_features:
    if infer_feature_type(X_train[col]) == "numeric":
        numeric_features.append(col)
    else:
        categorical_features.append(col)

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 7
Categorical features: 62


In [17]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ],
    remainder="drop"
)

In [18]:
baseline_model = DummyClassifier(strategy="most_frequent")

baseline_model.fit(X_train, y_train)
baseline_pred = baseline_model.predict(X_test)

print("Baseline precision:", precision_score(y_test, baseline_pred, zero_division=0))
print("Baseline recall:", recall_score(y_test, baseline_pred, zero_division=0))
print("Baseline f1:", f1_score(y_test, baseline_pred, zero_division=0))

Baseline precision: 0.0
Baseline recall: 0.0
Baseline f1: 0.0


In [19]:
logreg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ))
])

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

In [20]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "roc_auc": "roc_auc",
    "avg_precision": "average_precision",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall"
}

logreg_cv = cross_validate(logreg_pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
rf_cv = cross_validate(rf_pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)

cv_summary = pd.DataFrame({
    "model": ["logreg", "random_forest"],
    "roc_auc_mean": [logreg_cv["test_roc_auc"].mean(), rf_cv["test_roc_auc"].mean()],
    "avg_precision_mean": [logreg_cv["test_avg_precision"].mean(), rf_cv["test_avg_precision"].mean()],
    "f1_mean": [logreg_cv["test_f1"].mean(), rf_cv["test_f1"].mean()],
    "precision_mean": [logreg_cv["test_precision"].mean(), rf_cv["test_precision"].mean()],
    "recall_mean": [logreg_cv["test_recall"].mean(), rf_cv["test_recall"].mean()],
}).sort_values("avg_precision_mean", ascending=False)

cv_summary

,model,roc_auc_mean,avg_precision_mean,f1_mean,precision_mean,recall_mean
1,random_forest,0.710121,0.172358,0.068804,0.188269,0.042118
0,logreg,0.672307,0.169026,0.244138,0.157734,0.540300


In [21]:
best_pipeline = logreg_pipeline
best_model_name = "logreg"

best_pipeline.fit(X_train, y_train)

test_proba = best_pipeline.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.50).astype(int)

print("Best model:", best_model_name)
print("ROC-AUC:", round(roc_auc_score(y_test, test_proba), 4))
print("Average Precision:", round(average_precision_score(y_test, test_proba), 4))
print("Precision:", round(precision_score(y_test, test_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, test_pred, zero_division=0), 4))
print("F1:", round(f1_score(y_test, test_pred, zero_division=0), 4))
print()
print("Confusion Matrix:")
print(confusion_matrix(y_test, test_pred))
print()
print(classification_report(y_test, test_pred, zero_division=0))

Best model: logreg
ROC-AUC: 0.7038
Average Precision: 0.1824
Precision: 0.1614
Recall: 0.5956
F1: 0.2539

Confusion Matrix:
[[931 421]
 [ 55  81]]

              precision    recall  f1-score   support

           0       0.94      0.69      0.80      1352
           1       0.16      0.60      0.25       136

    accuracy                           0.68      1488
   macro avg       0.55      0.64      0.53      1488
weighted avg       0.87      0.68      0.75      1488



In [22]:
perm = permutation_importance(
    best_pipeline,
    X_test,
    y_test,
    n_repeats=20,
    random_state=RANDOM_STATE,
    scoring="average_precision",
    n_jobs=-1
)

perm_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

perm_df.head(30)

,feature,importance_mean,importance_std
52,mcq366b___doctor_told_to_increase_exercise,0.020931,0.007752
40,slq310___usual_wake_time_on_weekdays_or_workdays,0.006758,0.006636
36,mcq160m___ever_told_you_had_thyroid_problem,0.006041,0.003996
54,mcq366d___doctor_told_to_reduce_fat_in_diet,0.005743,0.005653
1,paq670___days_moderate_recreational_activities,0.005229,0.002250
19,dpq040___feeling_tired_or_having_little_energy,0.004052,0.003345
65,slq030___how_often_do_you_snore?,0.003328,0.003513
41,slq320___usual_sleep_time_on_weekends,0.003052,0.007231
11,alq130___avg_#_alcoholic_drinks/day___past_12_mos,0.002888,0.002739
7,cdq001___sp_ever_had_pain_or_discomfort_in_chest,0.002756,0.002824
